In [1]:
#00 中文處理
!wget "https://www.wfonts.com/download/data/2014/06/01/simhei/simhei.zip"
!unzip "simhei.zip"
!rm "simhei.zip"
from matplotlib.font_manager import FontProperties
myfont = FontProperties(fname=r'SimHei.ttf')

def plot_chinese(ax):
    labels = ax.get_xticklabels()+ax.legend().texts+[ax.title]+[ax.xaxis.get_label()]
    for label in labels :
        label.set_fontproperties(myfont)

--2024-04-30 06:48:29--  https://www.wfonts.com/download/data/2014/06/01/simhei/simhei.zip
Resolving www.wfonts.com (www.wfonts.com)... 104.21.1.127, 172.67.129.58, 2606:4700:3031::ac43:813a, ...
Connecting to www.wfonts.com (www.wfonts.com)|104.21.1.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10546298 (10M) [application/octetstream]
Saving to: ‘simhei.zip’

simhei.zip          100%[===================>]  10.06M  12.0MB/s    in 0.8s    

2024-04-30 06:48:30 (12.0 MB/s) - ‘simhei.zip’ saved [10546298/10546298]

Archive:  simhei.zip
  inflating: chinese.simhei.ttf      
  inflating: SimHei.ttf              
  inflating: sharefonts.net.txt      


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.datasets import load_iris
warnings.filterwarnings('ignore')

iris = load_iris()
iris.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [3]:
df = pd.DataFrame(iris['data'], columns=iris['feature_names'])
df['target'] = iris['target']
iris['feature_names']

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [5]:
X_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
y_col = 'target'
X = df[X_cols]
y = df[y_col]

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

1. 用鳶尾花全部的欄位來做羅吉斯迴歸模型預測。

(1) 請將資料做標準化，並輸出正確率、混亂矩陣和綜合報告。請注意，全部資料裡一共有三個類別的花朵。

In [7]:
#23
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

model_pl = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear'))
model_pl.fit(X_train, y_train)
y_pred = model_pl.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print('Accurancy: ', model_pl.score(X_test, y_test).round(2))

print('\nConfusion Matrix：\n')
print(pd.DataFrame(confusion_matrix(y_test, y_pred),
                   index=['實際1', '實際2', '實際3'], columns=['預測1', '預測2', '預測3']))
print('\nClassification Report：')
print(classification_report(y_test, y_pred))

Accurancy:  0.88

Confusion Matrix：

     預測1  預測2  預測3
實際1   19    0    0
實際2    0   10    5
實際3    0    1   15

Classification Report：
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       0.91      0.67      0.77        15
           2       0.75      0.94      0.83        16

    accuracy                           0.88        50
   macro avg       0.89      0.87      0.87        50
weighted avg       0.89      0.88      0.88        50



(2) 請預測這筆資料的結果是什麼？ [2,1,3,3]。（提示：請注意資料維度問題。）

In [14]:
model_pl.predict([[2, 1, 3, 3]])

array([1])

代表預測結果為第一種花（Versicolor）

(3) 請預測這兩筆資料的結果是什麼？ [2,1,3,3],[4,5,7,8]。（提示：請注意資料維度問題。）

In [15]:
model_pl.predict([[2, 1, 3, 3], [4 ,5, 7 ,8]])
#print(y_pred, "代表是第 1, 2 種花 versicolor 和 virginica")

array([1, 2])

代表預測結果為第一和第二種花（Versicolor、Virginica）

2. 請用本章的資料（即用兩個特徵值和 50 筆之後的資料）來做實驗。

我們希望讓預測為 2 的 Virginica 能有最大召回率，因此將機率判斷門檻設為0，請問：

In [10]:
df = pd.DataFrame(iris['data'], columns=iris['feature_names'])
df['target'] = iris['target']
df = df[['sepal width (cm)', 'petal length (cm)','target']]
df = df.iloc[50:]
df.head()

,sepal width (cm),petal length (cm),target
50,3.2,4.7,1
51,3.2,4.5,1
52,3.1,4.9,1
53,2.3,4.0,1
54,2.8,4.6,1


In [11]:
X_cols = ['sepal width (cm)', 'petal length (cm)']
y_col = 'target'
X = df[X_cols]
y = df[y_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
model_pl.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(solver='liblinear'))])

(1) Virginica 的召回率和精確率，混亂矩陣和綜合報表。請問你觀察到什麼現
象？

In [12]:
from sklearn.metrics import precision_score, recall_score
y_pred_proba = model_pl.predict_proba(X_test)[:,1]
y_pred_8 = np.where(y_pred_proba>=0, 2, 1)

scores = []
precision = precision_score(y_test, y_pred_8, pos_label=2)
recall = recall_score(y_test, y_pred_8, pos_label=2)
scores.append([0, precision, recall]) #分別設定 門檻(0)、精確率(precision)、召回率(recall)

print('Accurany: ', accuracy_score(y_test, y_pred_8).round(2))
print('\nConfusion Matrix: \n')
print(pd.DataFrame(confusion_matrix(y_test, y_pred_8), index=['實際1', '實際2'], columns=['預測1', '預測2']))
print('\nClassfication Report：\n')
print(classification_report(y_test, y_pred_8))

df_p_r = pd.DataFrame(scores, columns=['門檻','精確率','召回率'])
df_p_r.sort_values(by='門檻')

Accurany:  0.42

Confusion Matrix: 

     預測1  預測2
實際1    0   19
實際2    0   14

Classfication Report：

              precision    recall  f1-score   support

           1       0.00      0.00      0.00        19
           2       0.42      1.00      0.60        14

    accuracy                           0.42        33
   macro avg       0.21      0.50      0.30        33
weighted avg       0.18      0.42      0.25        33



,門檻,精確率,召回率
0,0,0.424242,1.0


召回率的意思是在所有實際為某類別的樣本中，被預測為該類別的比例，其公式為：Recall = TP/(TP+FN)。

而精確率的意思則是所有預測為某類別的樣本中，實際為該類別的比例，其公式為：Precision = TP/(TP+FP)。

以此題為例

*   召回率 = 實際2且預測2/(實際2且預測1+實際2且預測2)
*   精確率 = 實際2且預測2/(實際1且預測2+實際2且預測2)

在此題召回率=1，代表所有實際為第二類的花，皆正確預測成第二類。但精確度不為1，甚至很低，代表模型中存在很多FP，把非第二類的樣本預測為第二類（此題中，是把兩種花都預測到第二類裡）


(2) 若將 Virginica 的機率判斷門檻設為 0.99，請問其召回率和精確率，混亂矩陣和綜合報表。請問你觀察到什麼現象？

In [13]:
y_pred_8 = np.where(y_pred_proba>=0.99, 2, 1)

scores = []
prec = precision_score(y_test, y_pred_8, pos_label=2)
recall = recall_score(y_test, y_pred_8, pos_label=2)
scores.append([0.99, prec, recall])

print('Accurancy:', accuracy_score(y_test, y_pred_8).round(2))
print('\nConfusion Matrix: \n')
print(pd.DataFrame(confusion_matrix(y_test, y_pred_8), index=['實際1', '實際2'], columns=['預測1', '預測2']))
print('\n Classfication Report: \n')
print(classification_report(y_test, y_pred_8))

df_p_r = pd.DataFrame(scores, columns=['門檻','精確率','召回率'])
print("精確度=1 但 Recall 不好，代表 model 多有 FN。在此case中，常把第二類的花當作非第二類(第一類)的花")
df_p_r.sort_values(by='門檻')

Accurancy: 0.7

Confusion Matrix: 

     預測1  預測2
實際1   19    0
實際2   10    4

 Classfication Report: 

              precision    recall  f1-score   support

           1       0.66      1.00      0.79        19
           2       1.00      0.29      0.44        14

    accuracy                           0.70        33
   macro avg       0.83      0.64      0.62        33
weighted avg       0.80      0.70      0.64        33

精確度=1 但 Recall 不好，代表 model 多有 FN。在此case中，常把第二類的花當作非第二類(第一類)的花


,門檻,精確率,召回率
0,0.99,1.0,0.285714


呈2-(1)題解釋，在此題精確率=1，代表所有預測為第二類的花，實際皆為第二類。但召回率不為1，甚至很低，代表模型中存在很多FN，把第二類的樣本預測為第一類。